# Department of Computer Science & Engineering
## Machine Learning Laboratory (B.Tech)
### Assignment 02: Implementation of Binary and Multinomial Logistic Regression

---

## 1. Introduction

Logistic Regression is a foundational supervised classification algorithm used to predict categorical outcomes based on independent predictor variables.

* **Binary Logistic Regression:** Used when the target variable has exactly two discrete categories (e.g., 0 vs 1, Positive vs Negative). It utilizes the **Sigmoid (logistic) function** to map any real value into a probability range between 0 and 1.
* **Multinomial Logistic Regression:** An extension used when the target variable consists of three or more nominal categories (e.g., low, medium, high). It generalizes the logistic function to the **Softmax function** to compute probability distributions across multiple output classes.

**Objective of this Practical:**
1. To understand the mathematical principles and practical implementations of Binary and Multinomial Logistic Regression.
2. To preprocess continuous and categorical features using standard Scikit-Learn workflows.
3. To evaluate classification models using Accuracy, Precision, Recall, F1-Score, Classification Reports, and Confusion Matrices.
4. To visualize performance metrics using Confusion Matrix heatmaps, ROC-AUC curves, and class distribution plots.

### Google Colab Drive Mount (Optional for Colab Users)

In [ ]:
# Run this cell if running inside Google Colab to mount Drive and set directory
try:
    from google.colab import drive
    import os

    # Mount Google Drive
    drive.mount('/content/drive')

    # Change working directory to your assignment folder
    os.chdir('/content/drive/MyDrive/ML/Assignment_02_LogisticRegression')
    print("Current Working Directory:", os.getcwd())
except Exception as e:
    print("Not running in Colab or Drive already mounted:", e)

## 2. Import Required Libraries

In [ ]:
# Standard data handling and mathematical operations
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn preprocessing, models, and evaluation metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc
)

# Plotting configurations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

---
# PART A: Binary Logistic Regression

### Dataset Details
* **Dataset Name:** Diabetes Prediction Dataset
* **Kaggle Dataset Link:** [https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset](https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset)
* **Description:** This dataset contains patient demographic data, medical history, and clinical diagnostic measurements (age, gender, BMI, HbA1c level, blood glucose level, hypertension, heart disease) used to predict whether a patient has diabetes.
* **Why Binary Logistic Regression is Suitable:** The target column (`diabetes`) is strictly binary (0 = Non-diabetic, 1 = Diabetic) dependent on continuous medical factors, making it an ideal choice for binary logistic regression using the sigmoid activation function.

### A.1 Load the Binary Dataset

In [ ]:
# Relative path loading using os.path.join
binary_path = os.path.join("data", "binary_dataset.csv")

# Load dataset into pandas DataFrame
df_binary = pd.read_csv(binary_path)

print("Binary Dataset loaded successfully!")
print(f"Shape: {df_binary.shape[0]} samples, {df_binary.shape[1]} columns")

### A.2 Basic Data Exploration

In [ ]:
# Display first 5 rows
df_binary.head()

In [ ]:
# Check data types and non-null values
df_binary.info()

In [ ]:
# Statistical summary of numerical columns
df_binary.describe()

In [ ]:
# Check for missing values
print("Missing values in Binary Dataset:")
print(df_binary.isnull().sum())

### A.3 Data Preprocessing
We convert categorical variables (`gender`, `smoking_history`) into numeric representations using One-Hot Encoding (`pd.get_dummies()`).

In [ ]:
# Perform One-Hot Encoding for categorical features
df_binary_encoded = pd.get_dummies(df_binary, columns=["gender", "smoking_history"], drop_first=True)

# Display head of encoded dataframe
df_binary_encoded.head()

### A.4 Dataset Splitting & Feature Scaling

In [ ]:
# Separate features (X) and target (y)
X_b = df_binary_encoded.drop(columns=["diabetes"])
y_b = df_binary_encoded["diabetes"]

# Split into training (80%) and testing (20%) sets
X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
    X_b, y_b, test_size=0.2, random_state=42, stratify=y_b
)

# Apply StandardScaler for stable logistic regression convergence
scaler_b = StandardScaler()
X_b_train_scaled = scaler_b.fit_transform(X_b_train)
X_b_test_scaled = scaler_b.transform(X_b_test)

print(f"Binary Training Set shape: {X_b_train.shape}")
print(f"Binary Testing Set shape:  {X_b_test.shape}")

### A.5 Model Training & Predictions

In [ ]:
# Initialize and train Binary Logistic Regression model
binary_model = LogisticRegression(max_iter=1000, random_state=42)
binary_model.fit(X_b_train_scaled, y_b_train)

# Predict class labels and probabilities on test set
y_b_pred = binary_model.predict(X_b_test_scaled)
y_b_prob = binary_model.predict_proba(X_b_test_scaled)[:, 1]

print("Binary Model Training Complete!")

### A.6 Model Evaluation

In [ ]:
# Calculate metrics for Binary Logistic Regression
acc_b = accuracy_score(y_b_test, y_b_pred)
prec_b = precision_score(y_b_test, y_b_pred)
rec_b = recall_score(y_b_test, y_b_pred)
f1_b = f1_score(y_b_test, y_b_pred)

print("=" * 45)
print("   BINARY LOGISTIC REGRESSION METRICS")
print("=" * 45)
print(f"Accuracy  : {acc_b:.4f}")
print(f"Precision : {prec_b:.4f}")
print(f"Recall    : {rec_b:.4f}")
print(f"F1-Score  : {f1_b:.4f}")
print("=" * 45)
print("
Detailed Classification Report:")
print(classification_report(y_b_test, y_b_pred, target_names=["Non-Diabetic (0)", "Diabetic (1)"]))

### A.7 Binary Visualizations

In [ ]:
# Ensure output directory exists
os.makedirs("outputs", exist_ok=True)

# Visualization 1: Confusion Matrix Heatmap
cm_b = confusion_matrix(y_b_test, y_b_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_b, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Non-Diabetic", "Diabetic"],
            yticklabels=["Non-Diabetic", "Diabetic"])
plt.title("Binary Logistic Regression - Confusion Matrix", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.savefig(os.path.join("outputs", "binary_confusion_matrix.png"), dpi=300)
plt.show()

In [ ]:
# Visualization 2: ROC-AUC Curve
fpr, tpr, _ = roc_curve(y_b_test, y_b_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC Curve (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Random Classifier")
plt.title("Binary Logistic Regression - ROC Curve", fontsize=13, fontweight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join("outputs", "binary_roc_curve.png"), dpi=300)
plt.show()

In [ ]:
# Visualization 3: Target Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=y_b, palette="Set2")
plt.title("Target Distribution (Diabetes Outcome)", fontsize=13, fontweight="bold")
plt.xlabel("Outcome (0: Non-Diabetic, 1: Diabetic)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join("outputs", "binary_target_distribution.png"), dpi=300)
plt.show()

---
# PART B: Multinomial Logistic Regression

### Dataset Details
* **Dataset Name:** Mobile Price Classification Dataset
* **Kaggle Dataset Link:** [https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification](https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification)
* **Description:** This dataset contains technical specifications of mobile phones (RAM, battery power, internal memory, processor speed, screen dimensions, resolution, camera quality) to predict their price tier.
* **Why Multinomial Logistic Regression is Suitable:** The target attribute (`price_range`) consists of 4 distinct numerical categories (`0`: Low Cost, `1`: Medium Cost, `2`: High Cost, `3`: Very High Cost), which requires Softmax-based multinomial classification.

### B.1 Load the Multiclass Dataset

In [ ]:
# Relative path loading
multi_path = os.path.join("data", "multiclass_dataset.csv")

# Load dataset
df_multi = pd.read_csv(multi_path)

print("Multiclass Dataset loaded successfully!")
print(f"Shape: {df_multi.shape[0]} samples, {df_multi.shape[1]} columns")

### B.2 Basic Data Exploration

In [ ]:
# Display first 5 rows
df_multi.head()

In [ ]:
# Check data types and non-null values
df_multi.info()

In [ ]:
# Statistical summary
df_multi.describe()

In [ ]:
# Check missing values
print("Missing values in Multiclass Dataset:")
print(df_multi.isnull().sum())

### B.3 Data Preprocessing & Scaling
The features in this dataset are numerical. We separate features and target, then standardize feature magnitudes using `StandardScaler`.

In [ ]:
# Separate features (X) and target (y)
X_m = df_multi.drop(columns=["price_range"])
y_m = df_multi["price_range"]

# Split into training (80%) and testing (20%) sets
X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(
    X_m, y_m, test_size=0.2, random_state=42, stratify=y_m
)

# Standardize features for gradient descent optimization
scaler_m = StandardScaler()
X_m_train_scaled = scaler_m.fit_transform(X_m_train)
X_m_test_scaled = scaler_m.transform(X_m_test)

print(f"Multiclass Training Set shape: {X_m_train.shape}")
print(f"Multiclass Testing Set shape:  {X_m_test.shape}")

### B.4 Model Training & Predictions

In [ ]:
# Initialize and train Multinomial Logistic Regression model
multi_model = LogisticRegression(solver="lbfgs", max_iter=1000, random_state=42)
multi_model.fit(X_m_train_scaled, y_m_train)

# Make class predictions
y_m_pred = multi_model.predict(X_m_test_scaled)

print("Multinomial Model Training Complete!")

### B.5 Model Evaluation

In [ ]:
# Calculate multiclass metrics (using weighted average for multi-class)
acc_m = accuracy_score(y_m_test, y_m_pred)
prec_m = precision_score(y_m_test, y_m_pred, average="weighted")
rec_m = recall_score(y_m_test, y_m_pred, average="weighted")
f1_m = f1_score(y_m_test, y_m_pred, average="weighted")

print("=" * 45)
print("  MULTINOMIAL LOGISTIC REGRESSION METRICS")
print("=" * 45)
print(f"Accuracy  : {acc_m:.4f}")
print(f"Precision : {prec_m:.4f}")
print(f"Recall    : {rec_m:.4f}")
print(f"F1-Score  : {f1_m:.4f}")
print("=" * 45)
print("
Detailed Classification Report:")
target_names = ["Low Cost (0)", "Medium Cost (1)", "High Cost (2)", "Very High Cost (3)"]
print(classification_report(y_m_test, y_m_pred, target_names=target_names))

### B.6 Multinomial Visualizations

In [ ]:
# Visualization 1: Multiclass Confusion Matrix Heatmap
cm_m = confusion_matrix(y_m_test, y_m_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_m, annot=True, fmt="d", cmap="Purples", cbar=False,
            xticklabels=target_names, yticklabels=target_names)
plt.title("Multinomial Logistic Regression - Confusion Matrix", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Price Tier")
plt.ylabel("Actual Price Tier")
plt.tight_layout()
plt.savefig(os.path.join("outputs", "multinomial_confusion_matrix.png"), dpi=300)
plt.show()

In [ ]:
# Visualization 2: RAM vs Price Range Scatter Plot
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_multi, x="price_range", y="ram", palette="Set3")
plt.title("RAM Distribution Across Price Ranges", fontsize=13, fontweight="bold")
plt.xlabel("Price Range (0: Low, 1: Medium, 2: High, 3: Very High)")
plt.ylabel("RAM (MB)")
plt.tight_layout()
plt.savefig(os.path.join("outputs", "ram_vs_price_range.png"), dpi=300)
plt.show()

In [ ]:
# Visualization 3: Price Range Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=y_m, palette="Set2")
plt.title("Multiclass Dataset Price Range Distribution", fontsize=13, fontweight="bold")
plt.xlabel("Price Range Class")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join("outputs", "multiclass_target_distribution.png"), dpi=300)
plt.show()

---
# PART C: Comparison & Conclusion

### Comparison between Binary and Multinomial Logistic Regression

| Metric / Property | Binary Logistic Regression | Multinomial Logistic Regression |
| :--- | :--- | :--- |
| **Number of Target Classes** | Exactly 2 classes (Binary: 0 or 1) | 3 or more classes (Multi-class: 0, 1, 2, 3...) |
| **Activation Function** | Sigmoid $\sigma(z) = \frac{1}{1 + e^{-z}}$ | Softmax $\sigma(z)_i = \frac{e^{z_i}}{\sum e^{z_j}}$ |
| **Loss Function** | Binary Cross-Entropy (Log Loss) | Categorical Cross-Entropy |
| **Decision Boundary** | Single hyper-plane separating 2 regions | Multiple hyper-planes partitioning space into $K$ regions |
| **Output Probability** | Probability of belonging to Positive Class ($P(Y=1)$) | Probability distribution vector sum to 1 ($[P_0, P_1, ..., P_{K-1}]$) |
| **Typical Applications** | Spam Detection, Disease Diagnosis, Churn Prediction | Price Tier Grading, Customer Segmentation, Digit Recognition |

---

## Conclusion

1. **Binary Logistic Regression** was successfully evaluated on the Diabetes Prediction dataset, achieving high classification accuracy (~96%) and strong ROC-AUC performance using the Sigmoid function.
2. **Multinomial Logistic Regression** was effectively implemented on the Mobile Price Classification dataset to classify mobile phones into 4 distinct price ranges using the Softmax formulation, achieving ~96% accuracy.
3. Feature standardization (`StandardScaler`) proved essential for optimizing model convergence and model parameter stability in both tasks.
4. Confusion matrices and detailed classification reports confirmed that feature importance (such as `HbA1c_level` for diabetes and `ram` for mobile prices) strongly drives linear separation boundaries across targets.
5. Overall, this practical demonstrated the scalability of Logistic Regression from two-class binary decisions to complex multi-class decision problems.